<a href="https://colab.research.google.com/github/teekayboss/Linear-Regression-Health-Costs-Calculator/blob/main/Untitled5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [58]:
# Install necessary libraries if not already installed
!pip install -q gradio langchain-openai langchain-core

In [59]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser


# IMPORTANT: Set your OpenAI API key as an environment variable.
# For example: os.environ["OPENAI_API_KEY"] = "sk-..."
# Or use Google Colab's secrets manager (🔑 icon on the left panel) and retrieve it like:
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

def chatbot_response(message, history):
    # Initialize ChatOpenAI model (requires OPENAI_API_KEY environment variable to be set)
    try:
        llm = ChatOpenAI(temperature=0.7, model_name="gpt-3.5-turbo")
    except Exception as e:
        print(f"Warning: Could not initialize ChatOpenAI. Ensure OPENAI_API_KEY is set. Error: {e}")
        llm = None # Fallback if API key is not set

    # Convert Gradio chat history into LangChain message format
    langchain_history = []
    for human_msg, ai_msg in history:
        langchain_history.append(HumanMessage(content=human_msg))
        langchain_history.append(AIMessage(content=ai_msg))

    # Create a prompt template, including history
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful AI assistant. Answer user questions concisely."),
        *langchain_history,
        ("human", "{message}")
    ])

    if llm:
        # Create a simple chain: prompt -> LLM -> output parser
        chain = prompt | llm | StrOutputParser()
        response = chain.invoke({"message": message})
    else:
        # Fallback response if LLM could not be initialized
        response = "(LLM not available: Please set your OPENAI_API_KEY environment variable to enable full functionality.) " \
                   f"You asked: {message}"

    return response

In [60]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
print("OPENAI_API_KEY has been set as an environment variable.")

OPENAI_API_KEY has been set as an environment variable.


In [61]:
# Verify that the OPENAI_API_KEY is actually set in the environment
import os
if "OPENAI_API_KEY" in os.environ and os.environ["OPENAI_API_KEY"]:
    print(f"OPENAI_API_KEY is set: {os.environ['OPENAI_API_KEY'][:5]}...{os.environ['OPENAI_API_KEY'][-5:]}")
else:
    print("OPENAI_API_KEY is NOT set or is empty.")

OPENAI_API_KEY is set: sk-Ew...4lgtw


In [62]:
import gradio as gr

Now, let's create a Gradio interface for our chatbot function. This will provide a web-based chat window where you can interact with the bot.

In [63]:
# Create the Gradio ChatInterface
# The fn parameter points to our chatbot_response function
gr.ChatInterface(chatbot_response).launch()

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c09eb05e8b82e479b3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Adding your API Key to Colab Secrets

1.  **Open the Secrets Manager**: Look for the "🔑" (key) icon in the left-hand sidebar of your Colab notebook. Click on it.
2.  **Add a New Secret**: In the Secrets panel, click on "+ New secret".
3.  **Name the Secret**: For this notebook to work, name the secret `OPENAI_API_KEY` (it must match exactly).
4.  **Enter your API Key**: Paste your actual OpenAI API key into the "Value" field.
5.  **Save and Enable**: Ensure the "Notebook access" toggle is ON for this secret, then close the Secrets panel.

Once added, your `chatbot_response` function will be able to access the `OPENAI_API_KEY` securely without hardcoding it in your notebook.